# VietMedBridge — 03: Kiểm toàn bộ dữ liệu → freeze → chia input embedding

**Runtime CPU, Run all sau notebook 02.** Tự lấy đúng build từ active_data_build.json.
Kiểm mọi official ID/URL, source hash, offsets, parent/child và bảo toàn IDs lỗi.
Global dedup giữ toàn bộ aliases; input model giống hệt chỉ cần encode một lần.
Chuẩn bị input embedding thành các file nhỏ trên disk, chưa nạp model GPU.

Candidate freeze là mốc dữ liệu có integrity, không tự duyệt relevance hoặc
human QA. Source audit/golden thật vẫn cần team review; không tạo nhãn thi.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "data_processing_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
DATA_WORKFLOW_API = "external-extraction-import-v4"
upgrade = lock.get("workflow_api") != DATA_WORKFLOW_API
reference = CODE_REVISION or ("main" if upgrade else lock.get("git_commit")) or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION or upgrade:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION, "workflow_api": DATA_WORKFLOW_API})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "data_processing_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Đọc build hoàn tất và chạy regression của chunker


In [ ]:
from vietmedbridge.artifacts import read_json, atomic_json
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.golden import run_golden_suite
from vietmedbridge.health import health_report, freeze_candidate
from vietmedbridge.index_inputs import prepare_index_inputs, publish_data_handoff

BUILD_RUN_OVERRIDE = None  # dùng để chủ động đọc build cũ
active_path = DATA_ROOT / "active_data_build.json"
active = read_json(active_path) if active_path.exists() else {}
BUILD_RUN = BUILD_RUN_OVERRIDE or active.get("build_run", "stage-a-data-v2-restored")
BUILD_DIR = DATA_ROOT / "processed" / BUILD_RUN
BUILD = read_json(BUILD_DIR / "build.json")
if not BUILD.get("selected_range_complete"):
    raise RuntimeError("Build còn thiếu shard. Resume notebook 02 trước.")
if not BUILD_RUN_OVERRIDE and active and active["snapshot_sha256"] != BUILD["snapshot_sha256"]:
    raise ValueError("Active build pointer và snapshot không khớp.")
SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
CONFIG = read_json(BUILD_DIR / "config.json")
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(CONFIG["tokenizer"]["revision"])
CHUNK_CONFIG = ChunkConfig(**CONFIG["chunks"])
GOLDEN = run_golden_suite(CHECKOUT / "tests/golden/cases.json", TOKENIZER, TOKENIZER_SPEC, CHUNK_CONFIG)
if not GOLDEN["passed"]:
    raise RuntimeError("Golden regression fail; chưa freeze candidate.")
atomic_json(DATA_ROOT / "reports/golden_latest.json", GOLDEN)
print("Build:", BUILD_RUN, "| requested IDs:", BUILD["requested_input_records"])


## 2. Health report và source audit — CPU


In [ ]:
CRAWL_DIR = DATA_ROOT / "crawl" / BUILD["crawl_run"] if BUILD.get("crawl_run") else None
HEALTH = health_report(BUILD_DIR, official_links=OFFICIAL_LINKS, crawl_dir=CRAWL_DIR,
    work_dir=WORK_DIR, audit_size=PIPELINE_CONFIG["audit_size"])
print(json.dumps({k: HEALTH[k] for k in ("coverage", "documents", "chunks", "dedup")}, ensure_ascii=False, indent=2))
from IPython.display import HTML, display
display(HTML((BUILD_DIR / HEALTH["files"]["audit"]["path"]).read_text()))
print("Human QA pending:", BUILD_DIR / HEALTH["files"]["golden_candidates"]["path"])


## 3. Freeze và chia input index theo phần — CPU


In [ ]:
CANDIDATE = freeze_candidate(BUILD_DIR, HEALTH, golden_report=GOLDEN)
CANDIDATE_NAME = f"candidate-{CANDIDATE['candidate_manifest_sha256'][:16]}.json"
INPUTS = prepare_index_inputs(BUILD_DIR, CANDIDATE_NAME, TOKENIZER,
    work_dir=WORK_DIR, part_size=4096)
HANDOFF = publish_data_handoff(DATA_ROOT, BUILD_RUN, CANDIDATE, INPUTS)
print(json.dumps(HANDOFF, ensure_ascii=False, indent=2))
print("Model input parts:", BUILD_DIR / "index_inputs/units.json")
print("Tiếp theo: notebook 04 tự nhận candidate để benchmark GPU có giới hạn.")


Candidate lớn không được nạp vào catalog RAM của pilot. Notebook 04 sẽ
đọc một mẫu từ các input parts để đo BGE/Qwen/reranker và ghi chi phí,
trước khi triển khai inference/index toàn corpus. Nhãn/relevance score không
được suy từ throughput hoặc các kiểm tra integrity này.
